In [2]:
import yaml

# Load your data map
with open("DINO_map.yaml", "r") as file:
    variable_file_map = yaml.safe_load(file)["variable_file_map"]

# Define what you want to compute
metric_requirements = {
    "check_density": ["temperature", "salinity"],
    "temperature_500m_30NS_metric": ["temperature"],
    "ACC_Drake_metric_2": ["velocity_u", "ssh"],
}


In [8]:

GRANULARITY_ORDER = ["10d", "1m", "3m", "1y"]
def get_available_granularities(variable_file_map):
    all_grans = set()
    for entries in variable_file_map.values():
        for entry in entries:
            all_grans.add(entry["granularity"])
    return sorted(all_grans, key=GRANULARITY_ORDER.index)


In [71]:
# from granularity_simple import get_available_granularities
# Get maximum granularity for which all (most) metrics can be computed assuming resampling

import os

def get_maximum_granularity(variable_file_map, metric_requirements):
    """
    Find the maximum granularity that is available in the variable_file_map.
    """
    
    all_grans = get_available_granularities(variable_file_map)

    print(all_grans)


    # probably best to flatten all vars to functions. T, S, SSH, 
    # loop over all functions and compute unique.

    all_vars = set()

    for fn, vars_list in metric_requirements.items():

        for var in vars_list:
            all_vars.add(var)
    
    print(all_vars)

    cache = {}

    # all_grans = ["10d", "1m"]
    # all_vars = ["temperature"]

    for gran in all_grans:
        for var in all_vars:
            if (var, gran) in cache:
                continue

            # this finds if var at that granularity exists in variable_file_map
            entry = variable_file_map.get(var, [])
            for e in entry:
                # print(e)
                if e["granularity"] == gran:
                    file_path = e['file']
                    if os.path.exists(file_path):
                        cache[(var, gran)] = True
                        break

            # Now resample backwards
            grans = GRANULARITY_ORDER[:GRANULARITY_ORDER.index(gran)]
            print(grans)

            for finer_gran in reversed(grans):
                if (var, finer_gran) in cache:
                    # print("Can be resampled")
                    cache[(var, gran)] = True

    # print(cache)
    gran_fn_map = {} #  "gran" : [fn names]

    for gran in all_grans:
        for fn, vars in metric_requirements.items():
            # print(fn, vars)
            if gran not in gran_fn_map:
                gran_fn_map[gran] = []

            if all((v,gran) in cache for v in vars ):
                gran_fn_map[gran].append(fn)

    # print(gran_fn_map)
    # now get maximum granularity for which all metrics exist
    # print(all_grans)
    counts = [len(gran_fn_map[gran]) for gran in gran_fn_map]
    max_index = counts.index(max(counts))
    return all_grans[max_index]
        
get_maximum_granularity(variable_file_map, metric_requirements)



['10d', '1m', '3m']
{'temperature', 'ssh', 'salinity', 'velocity_u'}
[]
[]
[]
[]
['10d']
['10d']
['10d']
['10d']
['10d', '1m']
['10d', '1m']
['10d', '1m']
['10d', '1m']


'1m'